# Dzien 2 / Krok 4: FE gotowe na produkcje

**Input:** `checkpoints/01_merged.parquet`
**Output:** `checkpoints/04_pipeline.pkl`

Ten sam Feature Engineering co w Kroku 2, przepisany na:
- `df.pipe`: kazdy krok FE to czysta funkcja, latwa w testowaniu
- `FeatureEngineeringTransformer`: owijka na df.pipe z prawidlowym fit/transform
- `VolumeDecompTransformer`: dekompozycja trend/seasonal jako sklearn transformer
- sklearn `Pipeline`: surowe dane -> predykcja w jednym obiekcie

Pipeline startuje z surowych danych z bazy. Serwis produkcyjny robi:
`pipe.predict_proba(df_raw_order)[0, 1]` - zero dodatkowego kodu preprocessing.

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import joblib

df = pd.read_parquet("checkpoints/01_merged.parquet")
print(f"Zaladowano: {df.shape}")

Zaladowano: (95832, 27)


## Feature Engineering przez df.pipe

Kazda funkcja: przyjmuje DataFrame, zwraca DataFrame, nie modyfikuje oryginalu.
Latwe do testowania w izolacji i skladania w dowolnej kolejnosci.

In [2]:
def _add_delivery_features(df):
    feat = df.copy()
    feat["estimated_delivery_days"] = (
        feat["order_estimated_delivery_date"] - feat["order_purchase_timestamp"]
    ).dt.days.clip(0, 120)
    # Wolna akceptacja platnosci moze sygnalizowac problemy lub antyfrauda
    feat["approval_time_hours"] = (
        feat["order_approved_at"] - feat["order_purchase_timestamp"]
    ).dt.total_seconds().div(3600).clip(0, 48).fillna(24)
    return feat

def _add_time_features(df):
    feat = df.copy()
    ts = feat["order_purchase_timestamp"]
    feat["purchase_hour"]  = ts.dt.hour
    feat["purchase_dow"]   = ts.dt.dayofweek
    feat["purchase_month"] = ts.dt.month
    feat["is_weekend"]     = (feat["purchase_dow"] >= 5).astype(int)
    return feat

def _add_price_features(df):
    feat = df.copy()
    feat["price_log"]     = np.log1p(feat["price"])
    feat["freight_ratio"] = feat["freight_value"] / feat["price"].clip(lower=1)
    return feat

def _add_payment_features(df):
    feat = df.copy()
    feat["payment_installments"] = feat["payment_installments"].fillna(0)
    feat["is_installment"]       = (feat["payment_installments"] > 1).astype(int)
    return feat

# Ilustracja lancucha pipe - czytelny zapis kroków FE
def feature_engineering(df):
    return (df
            .pipe(_add_delivery_features)
            .pipe(_add_time_features)
            .pipe(_add_price_features)
            .pipe(_add_payment_features))

print("Funkcje df.pipe zdefiniowane.")

Funkcje df.pipe zdefiniowane.


## FeatureEngineeringTransformer: df.pipe jako sklearn transformer

Owijamy lancuch `df.pipe` w sklearn transformer.

`fit()` zapamietuje tylko to co jest stanowe: enkodery kategorii i mediane wagi produktu.
Cztery funkcje `_add_*` sa bezstanowe — dzialaja tak samo na train i na nowych danych.

Wazne: LabelEncoder nauczony na train. Nieznane kategorie w produkcji (nowe stany, nowe typy platnosci)
sa mapowane na `"unknown"` zamiast rzucac bledem.

In [3]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    """
    Owijka na lancuch df.pipe.

    fit:       zapamietuje LabelEncoder per kolumna kategorialna + mediane wagi
    transform: aplikuje wszystkie kroki FE, uzywa zapamiętanych enkoderoów
    """

    CAT_COLS = ["payment_type", "product_category_name_english",
                "customer_state", "seller_state"]

    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        df_base = feature_engineering(df)

        self.label_encoders_ = {}
        for col in self.CAT_COLS:
            le = LabelEncoder()
            # Wlacz "unknown" do treningu - wtedy transform() moze go bezpiecznie uzywac
            # dla nieznanych kategorii z produkcji (nowe stany, nowe typy platnosci)
            values = pd.concat([
                df_base[col].fillna("unknown").astype(str),
                pd.Series(["unknown"])
            ]).drop_duplicates()
            le.fit(values)
            self.label_encoders_[col] = le

        self.weight_median_ = df_base["product_weight_g"].median()
        return self

    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        feat = feature_engineering(df)

        for col, le in self.label_encoders_.items():
            vals = feat[col].fillna("unknown").astype(str)
            # Nieznane kategorie -> "unknown" (bezpiecznie dla produkcji)
            known = set(le.classes_)
            vals  = vals.map(lambda v: v if v in known else "unknown")
            feat[col + "_enc"] = le.transform(vals)

        feat["product_weight_g"]   = feat["product_weight_g"].fillna(self.weight_median_)
        feat["product_photos_qty"] = feat["product_photos_qty"].fillna(1)
        return feat

## VolumeDecompTransformer: dekompozycja jako sklearn transformer

W Kroku 2 robilibysmy dekompozycje recznie na pelnym zbiorze przed splitem.
Tutaj pokazujemy wlasciwy sposob: transformer, ktory `fit`-uje sie tylko na danych
treningowych i zachowuje lookup w pamieci obiektu.

```
fit(X_train):
    1. Oblicz miesięczny wolumen zamowien z X_train
    2. seasonal_decompose -> trend + seasonal per miesiac
    3. Zapisz self.decomp_lookup_ (Period -> wartosci)

transform(X):
    1. Wyciagnij rok-miesiac z timestampa
    2. Zmapuj trend i seasonal z self.decomp_lookup_
    3. Zwroc DataFrame z dodanymi kolumnami
```

Dla miesiecy spoza zakresu treningowego (np. przyszlosc): trend interpolowany
liniowo przez `limit_direction="both"` podczas fitu.

In [4]:
class VolumeDecompTransformer(BaseEstimator, TransformerMixin):
    """
    Oblicza dekompozycje sezonowa miesięcznego wolumenu zamowien.

    fit:       uczy sie na szeregu z X (timestamps)
    transform: mapuje trend i sezonowosc na kazdy wiersz po month-period

    Parametr decomp_lookup: jesli podany, fit() uzywa gotowego lookup zamiast
    przeliczac dekompozycje od nowa. Przydatne gdy zbior treningowy jest zbyt
    krotki dla seasonal_decompose (wymaga >= 2 * period miesiecy).
    """

    def __init__(self, ts_col="order_purchase_timestamp", period=12,
                 decomp_lookup=None):
        self.ts_col        = ts_col
        self.period        = period
        self.decomp_lookup = decomp_lookup  # opcjonalny pre-computed lookup

    def fit(self, X, y=None):
        if self.decomp_lookup is not None:
            # Lookup dostarczony z zewnatrz - nie przeliczaj
            self.decomp_lookup_ = self.decomp_lookup
            return self

        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        monthly = (df.set_index(self.ts_col)
                    .resample("ME")
                    .size()
                    .rename("volume"))

        if len(monthly) < 2 * self.period:
            raise ValueError(
                f"Za malo miesiecy w danych treningowych: {len(monthly)} "
                f"(wymagane >= {2 * self.period} dla period={self.period}). "
                f"Uzyj VolumeDecompTransformer(decomp_lookup=lookup) "
                f"z lookup obliczonym na pelnym datasecie."
            )

        result = seasonal_decompose(monthly, model="additive", period=self.period)

        self.decomp_lookup_ = pd.DataFrame({
            "volume_trend":    result.trend.interpolate(
                                   method="linear", limit_direction="both"),
            "volume_seasonal": result.seasonal,
        })
        self.decomp_lookup_.index = self.decomp_lookup_.index.to_period("M")
        return self

    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        df = df.copy()
        periods = pd.to_datetime(df[self.ts_col]).dt.to_period("M")
        df["volume_trend"]    = periods.map(self.decomp_lookup_["volume_trend"])
        df["volume_seasonal"] = periods.map(self.decomp_lookup_["volume_seasonal"])
        return df

In [5]:
TARGET = "is_bad_experience"
y      = df[TARGET]
X_raw  = df.drop(columns=[TARGET])

X_raw_train, X_raw_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# VolumeDecompTransformer potrzebuje pelnego zakresu czasowego (>= 24 miesiecy).
# Fitujemy go na X_raw (pełny zbiór bez targetu).
vdt_ref = VolumeDecompTransformer()
vdt_ref.fit(X_raw)

print(f"Train: {len(X_raw_train):,}  |  Test: {len(X_raw_test):,}")
print(f"Lookup zbudowany na {len(vdt_ref.decomp_lookup_)} miesiacach")
print(f"Zakres: {vdt_ref.decomp_lookup_.index.min()} - {vdt_ref.decomp_lookup_.index.max()}")

Train: 76,665  |  Test: 19,167
Lookup zbudowany na 24 miesiacach
Zakres: 2016-09 - 2018-08


## Pelny Pipeline

Teraz Pipeline startuje bezposrednio z surowych danych - tych samych co laduje serwis produkcyjny.

```
X_raw (surowy DataFrame z bazy)
    -> FeatureEngineeringTransformer   # df.pipe: delivery, czas, ceny, platnosci, enkodery
    -> VolumeDecompTransformer         # dodaje volume_trend, volume_seasonal
    -> FeatureSelector                 # wybiera tylko FEATURE_COLS
    -> StandardScaler                  # normalizacja
    -> XGBClassifier                   # predykcja
```

Jeden obiekt `pipe` zastepuje caly preprocessing kod w serwisie.

In [6]:
FEATURE_COLS = [
    # Czas
    "estimated_delivery_days", "approval_time_hours",
    "purchase_hour", "purchase_dow", "purchase_month", "is_weekend",
    # Trend i sezonowosc (dodawane przez VolumeDecompTransformer)
    "volume_trend", "volume_seasonal",
    # Produkt i platnosc
    "price_log", "freight_ratio",
    "items_count", "payment_installments", "is_installment",
    "product_weight_g", "product_photos_qty",
    # Kategorie (zakodowane)
    "payment_type_enc", "product_category_name_english_enc",
    "customer_state_enc", "seller_state_enc",
]

class FeatureSelector(BaseEstimator, TransformerMixin):
    """Wybiera kolumny z DataFrame po transformacjach poprzednich krokow."""
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        return df[self.cols]

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()

pipe = Pipeline([
    ("fe",       FeatureEngineeringTransformer()),
    ("decomp",   VolumeDecompTransformer(decomp_lookup=vdt_ref.decomp_lookup_)),
    ("selector", FeatureSelector(FEATURE_COLS)),
    ("scaler",   StandardScaler()),
    ("model",    xgb.XGBClassifier(
        objective        = "binary:logistic",
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        scale_pos_weight = float(scale_pos_weight),
        eval_metric      = "auc",
        random_state     = 42,
        n_jobs           = -1,
    )),
])

pipe.fit(X_raw_train, y_train)
auc = roc_auc_score(y_test, pipe.predict_proba(X_raw_test)[:, 1])
print(f"Pipeline AUC: {auc:.4f}")
print(f"Kroki: {[name for name, _ in pipe.steps]}")

Pipeline AUC: 0.6710
Kroki: ['fe', 'decomp', 'selector', 'scaler', 'model']


In [7]:
# Inferens na jednym przykladzie - tak wyglada produkcja
single_order = X_raw_test.iloc[[0]]
proba = pipe.predict_proba(single_order)[0, 1]
decision = "RYZYKO - wyslij voucher" if proba > 0.5 else "OK"
print(f"Prawdopodobienstwo zlej recenzji: {proba:.3f}")
print(f"Decyzja (prog=0.5): {decision}")
print()
print("W serwisie produkcyjnym: ten sam obiekt 'pipe', te same transformacje.")
print("Jeden joblib.load() i gotowe.")

Prawdopodobienstwo zlej recenzji: 0.429
Decyzja (prog=0.5): OK

W serwisie produkcyjnym: ten sam obiekt 'pipe', te same transformacje.
Jeden joblib.load() i gotowe.


## Zapis Pipeline

In [1]:
pipeline_path =  "checkpoints/04_pipeline.pkl"
joblib.dump(pipe,pipeline_path)
print(f"Rozmiar pliku: {__import__('os').path.getsize(pipeline_path) / 1024:.0f} KB")

NameError: name 'joblib' is not defined